# **Индексы взаимодействия Шепли: SII, k-SII, STII, FSII и пакет shapiq**

Практика к модулю [«Взаимодействие признаков»](https://ai-interpretability.school), урок «Значения SHAP в задаче оценки взаимодействия», шаг «Конкретные индексы: от SII к высшим порядкам».

В уроке индексы взаимодействия были весами в общем каркасе: SII не эффективен на группах, k-SII, STII и FSII чинят это каждый по-своему, и на одной игре они дают разные числа. Здесь мы посчитаем их все пакетом [shapiq](https://github.com/mmschlk/shapiq) ([Muschalik и др., 2024](https://arxiv.org/abs/2410.01649)):

1. игра из трех игроков с известными дивидендами Мебиуса: четыре индекса и проверка эффективности;
2. модель спроса на прокат велосипедов и парные SHAP interaction values из `shap` рядом с SII из shapiq;
3. четыре индекса на одном объекте и разные функции ценности коалиции;
4. взаимодействия третьего порядка;
5. приближенный расчет: ошибка аппроксиматоров shapiq в зависимости от бюджета;
6. глобальная картина: средний модуль k-SII по выборке;
7. где ломается: коррелированные признаки.

Тетрадь идет на процессоре; весь прогон занимает две-три минуты.

In [ ]:
!pip install -q shapiq==1.7.0 shap==0.52.0

In [ ]:
import warnings
warnings.filterwarnings('ignore', message='IProgress not found')   # tqdm без ipywidgets; в Colab ipywidgets есть
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import shapiq
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore', message='.*encountered in matmul')   # ложные предупреждения numpy на некоторых процессорах
SEED = 0
pd.set_option('display.width', 140)

## 1. Игра с известными дивидендами

Дивиденды Мебиуса (дивиденды Харсаньи) $m(S)$ — то, что коалиция $S$ приносит сверх всех своих подкоалиций; любая игра раскладывается по ним: $v(T)=\sum_{S\subseteq T}m(S)$. Зададим игру трех игроков прямо дивидендами:

$$m(\{0\})=1,\quad m(\{1\})=1,\quad m(\{0,1\})=2,\quad m(\{0,1,2\})=3,$$

остальные дивиденды нулевые. Игрок 2 сам по себе ничего не дает и участвует только в тройном взаимодействии. Полный выигрыш $v(N)-v(\varnothing)=1+1+2+3=7$.

Индексы второго порядка должны как-то разделить тройной дивиденд 3 между парами и одиночками — и делят его по-разному. `shapiq.ExactComputer` считает любой индекс полным перебором коалиций.

In [ ]:
def toy_game(coalitions):
    # coalitions: булева матрица (число коалиций, 3); ценность — сумма дивидендов вошедших подмножеств
    c = coalitions.astype(float)
    return c[:, 0] + c[:, 1] + 2 * c[:, 0] * c[:, 1] + 3 * c[:, 0] * c[:, 1] * c[:, 2]


exact_toy = shapiq.ExactComputer(toy_game, n_players=3)
toy = {idx: exact_toy(index=idx, order=2) for idx in ('SII', 'k-SII', 'STII', 'FSII')}
subsets = [(0,), (1,), (2,), (0, 1), (0, 2), (1, 2)]
toy_tab = pd.DataFrame({idx: [iv[s] for s in subsets] for idx, iv in toy.items()},
                       index=[str(s) for s in subsets])
print(toy_tab.round(3))
print('дивиденды Мебиуса:', {k: round(v, 3) for k, v in exact_toy(index='Moebius', order=3).dict_values.items() if abs(v) > 1e-9})

**Задание 1. Для каждого из четырех индексов посчитайте сумму всех значений порядков 1 и 2 (шесть чисел в столбце таблицы) и сравните с полным выигрышем 7. Какой индекс не эффективен? Затем посчитайте k-SII третьего порядка (`exact_toy(index='k-SII', order=3)`) и сравните его значения с дивидендами Мебиуса.**

In [ ]:
# Ваш код здесь

Как четыре индекса делят тройной дивиденд: STII отдает младшим порядкам ровно их дивиденды, а весь тройной дивиденд распределяет поровну по трем парам; k-SII и FSII отдают парам по половине тройного дивиденда, и на одиночки приходится поправка, которая делает разложение эффективным; SII отдает парам то же, что k-SII, но одиночки у него — обычные значения Шепли, и сумма не сходится.

## 2. Модель спроса на велосипеды

[Bike Sharing](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset) (Fanaee-T и Gama, 2013): 17 379 часов работы проката Capital Bikeshare в Вашингтоне в 2011–2012 годах, исход — число поездок за час. Берем выгрузку из репозитория shapiq. Признаки: час суток `hour`, рабочий ли день `workingday`, температура `temp` и ощущаемая температура `feel_temp` (°C), влажность, ветер, погода, сезон, месяц, день недели, праздник, год. Предсказываем логарифм числа поездок случайным лесом.

Взаимодействие здесь известно заранее: в рабочие дни спрос имеет два пика, утром и вечером, а в выходные — один, днем. Значит, эффект часа зависит от того, рабочий ли день.

In [ ]:
bike = pd.read_csv('https://raw.githubusercontent.com/mmschlk/shapiq/main/data/bike.csv')
for c in ('season', 'weather'):
    bike[c] = bike[c].astype('category').cat.codes
y = np.log1p(bike.pop('count').values)
names = list(bike.columns)
X = bike.values.astype(float)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=SEED)
rf = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=SEED, n_jobs=-1).fit(Xtr, ytr)
print('признаки:', names)
print(f'R² на отложенной части: {rf.score(Xte, yte):.3f}')
H, W, TEMP, FEEL = (names.index(n) for n in ('hour', 'workingday', 'temp', 'feel_temp'))

hours = np.arange(24)
for wd, color in ((1, '#3b6ea5'), (0, '#e67e22')):
    m = X[:, W] == wd
    plt.plot(hours, [np.expm1(y[m & (X[:, H] == h)]).mean() for h in hours], 'o-', color=color,
             label='working day' if wd else 'day off')
plt.xlabel('Hour of day')
plt.ylabel('Mean rides per hour')
plt.title('Bike sharing: demand by hour')
plt.legend()
plt.show()

## Парные SHAP interaction values и SII

Урок говорит: парный $\phi_{ij}$ из `shap` — это симметризованный SII, $\phi_{ij}=\tfrac12\mathrm{SII}_{ij}$. Проверим на одном часе отложенной части: `shap.TreeExplainer(rf).shap_interaction_values` и `shapiq.TreeExplainer(..., index='SII', max_order=2)` — оба точные алгоритмы для деревьев с одной и той же функцией ценности коалиции (path-dependent: недостающие признаки «выключаются» по долям обучающих объектов в ветвях дерева).

Возьмем рабочий день, 8 утра — утренний пик.

In [ ]:
i0 = np.flatnonzero((Xte[:, W] == 1) & (Xte[:, H] == 8))[0]
x0 = Xte[i0]
print('выбранный час:', {n: float(v) for n, v in zip(names, x0)})
print(f'прогноз модели: {rf.predict(x0[None])[0]:.3f} (логарифм числа поездок)')

phi_pairs = shap.TreeExplainer(rf).shap_interaction_values(x0[None])[0]       # матрица M×M
sii_tree = shapiq.TreeExplainer(model=rf, index='SII', max_order=2).explain(x0)

**Задание 2. Для всех 66 пар признаков $(i, j)$, $i<j$, посчитайте наибольшее по модулю расхождение между `2 * phi_pairs[i, j]` и `sii_tree[(i, j)]`. Проверьте и второе свойство из урока: сумма строки $i$ матрицы `phi_pairs` равна обычному значению Шепли признака $i$ (возьмите `shap.TreeExplainer(rf).shap_values`). Выведите пару с наибольшим по модулю SII и ее значение.**

In [ ]:
# Ваш код здесь

## 3. Четыре индекса на одном объекте и функция ценности коалиции

Деревянный алгоритм shapiq считает не все индексы. Для сравнения всех четырех построим игру явно: ценность коалиции $S$ — средний прогноз леса, когда признаки из $S$ взяты у нашего часа, а остальные — у 100 случайных часов обучающей части (`shapiq.MarginalImputer`, интервенционная функция ценности). Двенадцать признаков — это $2^{12}=4096$ коалиций, и `ExactComputer` перебирает их все за несколько секунд.

Это **другая** функция ценности, чем path-dependent в деревянном алгоритме, поэтому SII из раздела 2 и SII здесь — разные числа одного индекса.

In [ ]:
background = Xtr[np.random.default_rng(SEED).choice(len(Xtr), 100, replace=False)]
game = shapiq.MarginalImputer(rf.predict, data=background, x=x0, sample_size=100, random_state=SEED)
exact = shapiq.ExactComputer(game, n_players=len(names))
ivs = {idx: exact(index=idx, order=2) for idx in ('SII', 'k-SII', 'STII', 'FSII')}
full = rf.predict(x0[None])[0] - rf.predict(background).mean()
print(f'полный выигрыш v(N) − v(∅) = прогноз − средний прогноз на фоне: {full:.4f}')
for idx, iv in ivs.items():
    print(f'{idx:>6}: сумма порядков 1 и 2 = {sum(v for k, v in iv.dict_values.items() if len(k) > 0):.4f}')

top_pairs = sorted(pairs, key=lambda p: -abs(ivs['k-SII'][p]))[:6]
cmp = pd.DataFrame({idx: [iv[p] for p in top_pairs] for idx, iv in ivs.items()},
                   index=[f'{names[i]} × {names[j]}' for i, j in top_pairs])
cmp['SII, path-dependent (раздел 2)'] = [sii_tree[p] for p in top_pairs]
print(cmp.round(4))

Пары у четырех индексов почти совпадают: в этой модели взаимодействия третьего порядка слабее парных (раздел 4), и тройным дивидендам нечего перераспределять. Одиночные значения расходятся сильнее — у SII это обычные значения Шепли, у остальных — эффекты, из которых вычтена доля взаимодействий. Картинка ниже — одиночные значения четырех индексов для шести признаков с наибольшим по модулю k-SII.

In [ ]:
top_single = sorted(range(len(names)), key=lambda i: -abs(ivs['k-SII'][(i,)]))[:6]
width = 0.2
fig, ax = plt.subplots(figsize=(10, 3.8))
for k, (idx, color) in enumerate(zip(ivs, ['#7f8c8d', '#3b6ea5', '#e67e22', '#27ae60'])):
    ax.bar(np.arange(6) + (k - 1.5) * width, [ivs[idx][(i,)] for i in top_single], width, color=color, label=idx)
ax.set_xticks(range(6))
ax.set_xticklabels([names[i] for i in top_single])
ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel('Order-1 value')
ax.set_title('Working day, 8 a.m.: singleton values of four order-2 indices')
ax.legend()
plt.show()

## 4. Взаимодействия третьего порядка

`shapiq.TreeExplainer` с `index='k-SII'` и `max_order=3` дает разложение прогноза до троек. Для k-SII сумма всех значений порядков 1–3 снова равна прогнозу минус базовое значение.

In [ ]:
k3_tree = shapiq.TreeExplainer(model=rf, index='k-SII', max_order=3).explain(x0)
by_order = {o: sum(v for k, v in k3_tree.dict_values.items() if len(k) == o) for o in (1, 2, 3)}
print('сумма значений по порядкам:', {o: round(v, 4) for o, v in by_order.items()})
print(f'прогноз − базовое значение: {rf.predict(x0[None])[0] - k3_tree.baseline_value:.4f}')
triples = sorted([k for k in k3_tree.dict_values if len(k) == 3], key=lambda k: -abs(k3_tree[k]))[:5]
for k in triples:
    print(f'{" × ".join(names[i] for i in k):>40}: {k3_tree[k]:+.4f}')

## 5. Приближенный расчет: ошибка в зависимости от бюджета

Перебор $2^M$ коалиций возможен при двенадцати признаках, но не при пятидесяти. Аппроксиматоры shapiq оценивают индекс по `budget` вычислениям игры. Сравним четыре из них с точным k-SII второго порядка из раздела 3 на той же игре: `KernelSHAPIQ` (взвешенная регрессия по коалициям), `SVARMIQ` (стратифицированная выборка), `SHAPIQ` (несмещенная выборка коалиций) и `PermutationSamplingSII` (перестановки). Ошибка — средний квадрат отклонения по 66 парам.

**Задание 3. Для бюджетов 128, 256, 512, 1024, 2048 и 4096 и каждого аппроксиматора посчитайте среднеквадратичную ошибку по парам относительно `ivs['k-SII']` (сид `random_state=SEED`, `max_order=2`, `index='k-SII'`; у `PermutationSamplingSII` тоже `index='k-SII'`). Нарисуйте ошибку от бюджета в логарифмических осях. Какой аппроксиматор точнее всех при бюджете 256?**

In [ ]:
# Ваш код здесь

При малом бюджете (128 и 256 вычислений игры) точнее всех перестановочная выборка, начиная с 512 — KernelSHAPIQ. При бюджете 4096 все аппроксиматоры, кроме перестановочного, видят все коалиции и выдают точное значение: ошибка падает до машинного нуля. Перестановочная выборка не умеет перебрать все коалиции и с ростом бюджета сходится медленнее остальных.

## 6. Глобальная картина

Индексы отвечают про один объект. Сводку по данным дает среднее модулей: средний $|\text{k-SII}|$ каждой пары по 300 часам отложенной части, посчитанный точным деревянным алгоритмом.

In [ ]:
tree_k2 = shapiq.TreeExplainer(model=rf, index='k-SII', max_order=2)
sample = Xte[np.random.default_rng(SEED).choice(len(Xte), 300, replace=False)]
mean_abs = pd.Series(0.0, index=[f'{names[i]} × {names[j]}' for i, j in pairs])
for xs in sample:
    iv = tree_k2.explain(xs)
    mean_abs += np.array([abs(iv[p]) for p in pairs]) / len(sample)
top = mean_abs.sort_values().tail(10)
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(top.index, top.values, color='#3b6ea5')
ax.set_xlabel('Mean |k-SII| over 300 held-out hours')
ax.set_title('Strongest pairwise interactions of the bike demand model')
plt.tight_layout()
plt.show()
print(top[::-1].round(4))

## 7. Где ломается: коррелированные признаки

`temp` и `feel_temp` почти дублируют друг друга. Модель может опираться на любой из них, и взаимодействие «час × температура» делится между двумя парами — «час × temp» и «час × feel_temp» — так, как решил лес, а не так, как устроен мир. Проверим: обучим тот же лес без `feel_temp` и сравним средний $|\text{k-SII}|$ пар с часом.

**Задание 4. Посчитайте корреляцию `temp` и `feel_temp` на обучающей части. Обучите лес с теми же гиперпараметрами без столбца `feel_temp` и посчитайте средний $|\text{k-SII}|$ пары `hour × temp` по тем же 300 часам `sample`. Сравните с суммой средних $|\text{k-SII}|$ пар `hour × temp` и `hour × feel_temp` полной модели.**

In [ ]:
# Ваш код здесь

**Задание 5. По выводам разделов 2, 3 и 7: назовите три выбора, которые надо указать рядом с числом индекса взаимодействия, чтобы это число можно было истолковать.**

**Ваш ответ:**

## Что унести из тетради

- **Индексы различаются распределением взаимодействий высших порядков.** На игре с тройным дивидендом SII, k-SII, STII и FSII дают разные одиночные и парные значения, а SII не сходится к полному выигрышу.
- **$\phi_{ij}$ из `shap` — это половина SII.** Оба точных деревянных алгоритма дают одно и то же с точностью до машинного нуля, и строки матрицы `shap` складываются в значения Шепли.
- **Функция ценности коалиции меняет числа сильнее, чем выбор индекса.** Path-dependent и интервенционная игра дают разные значения одного индекса на одном объекте.
- **Приближенный расчет требует бюджета, и лучший аппроксиматор зависит от него.** На двенадцати признаках ошибки аппроксиматоров shapiq при одном бюджете различаются на два порядка; при малом бюджете точнее перестановочная выборка, при большом — KernelSHAPIQ.
- **Коррелированные признаки делят взаимодействие.** Число у пары зависит от того, какие еще признаки видела модель.

Источники:
1. [Muschalik et al., 2024. shapiq: Shapley interactions for machine learning](https://arxiv.org/abs/2410.01649)
2. [Bordt & von Luxburg, 2023. From Shapley values to generalized additive models and back](https://proceedings.mlr.press/v206/bordt23a.html)
3. [Sundararajan et al., 2020. The Shapley Taylor interaction index](https://arxiv.org/abs/1902.05622)
4. [Tsai, Yeh & Ravikumar, 2023. Faith-Shap: the faithful Shapley interaction index](https://arxiv.org/abs/2203.00870)
5. [Fanaee-T & Gama, 2013. Event labeling combining ensemble detectors and background knowledge](https://doi.org/10.1007/s13748-013-0040-3) — данные Bike Sharing